In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PREFIX = "inertial-6286.188861:"

def read_csv(csv_path_file):
    buff = []
    with open(csv_path_file, "r") as file:
    # with open("CB_ARW.csv", "r") as file:
        data = False
        for line in file:
            
            if line == "DATA_START\n":
                data = True
                continue
            
            if data == True:
                line = line.rstrip()
                values = line.split(",")
                buff.append(values)

    header = [b.lstrip(PREFIX) for b in buff[0]]

    buff = np.array(buff)

    df = pd.DataFrame(data=buff[1:], columns=header)
    
    cols_to_drop = [c for c in df.columns if c.endswith('valid')]
    df = df.drop(columns=cols_to_drop)

    # Conversion de tipo de datos de df (string a numerico)
    for column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    return df

fs = 100
dt = 1 / fs


In [2]:
df_cuat = read_csv(r"calibracion_orientacion\cuat_4_est.csv")

df_imu = read_csv(r"calibracion_orientacion\acel_4_est.csv")

In [3]:
# Renombrar columnas
cuat_new_names = ["q0", "q1", "q2", "q3"]
df_cuat = df_cuat.rename(columns=dict(zip(df_cuat.columns[1:], cuat_new_names)))

# Tiempo relativo en segundos
df_cuat["t"] = (df_cuat["Time"] - df_cuat["Time"].iloc[0]) / 1e9

# Delta de tiempo entre muestras
df_cuat["dt"] = df_cuat["t"].diff()

# Reordenar columnas
df_cuat = df_cuat.reindex(columns=['Time', 't', 'dt', 'q0', 'q1', 'q2', 'q3'])

df_cuat

,Time,t,dt,q0,q1,q2,q3
0,1778882044327970304,0.000000,NaN,0.984476,-0.008141,-0.166982,0.053449
1,1778882044337928704,0.009958,0.009958,0.984475,-0.008156,-0.166987,0.053454
2,1778882044347867392,0.019897,0.009939,0.984475,-0.008152,-0.166986,0.053464
3,1778882044357786368,0.029816,0.009919,0.984476,-0.008139,-0.166986,0.053453
4,1778882044367685888,0.039716,0.009900,0.984477,-0.008143,-0.166981,0.053452
...,...,...,...,...,...,...,...
944,1778882053767425536,9.439455,0.010173,0.984448,-0.008012,-0.167037,0.053824
945,1778882053777578240,9.449608,0.010153,0.984448,-0.008013,-0.167036,0.053825
946,1778882053787710720,9.459740,0.010132,0.984448,-0.008013,-0.167037,0.053824
947,1778882053797822976,9.469853,0.010112,0.984447,-0.008016,-0.167038,0.053825


In [4]:
al_new_names = ["acx", "acy", "acz"]

df_imu = df_imu.rename(columns=dict(zip(df_imu.columns[1:], al_new_names)))

# Reordenar columnas
df_imu = df_imu.reindex(columns=['Time', 'acx', 'acy', 'acz'])

df_imu

,Time,acx,acy,acz
0,1778882044325243904,-0.328474,0.030808,-0.944354
1,1778882044335207936,-0.325551,0.033112,-0.940716
2,1778882044345152256,-0.325238,0.036366,-0.935931
3,1778882044355076608,-0.327034,0.033377,-0.937326
4,1778882044364980992,-0.327513,0.033002,-0.942016
...,...,...,...,...
944,1778882053764549120,-0.326244,0.033626,-0.939517
945,1778882053774707712,-0.326059,0.033698,-0.939841
946,1778882053784845824,-0.325983,0.033813,-0.939869
947,1778882053794963968,-0.326069,0.033642,-0.939342


In [5]:
tol_ns = 5_000_000  # 5 ms

df_u = pd.merge_asof(
    df_cuat,
    df_imu,
    on="Time",
    direction="nearest",
    tolerance=tol_ns
)

# Eliminar filas donde no hubo coincidencia dentro de la tolerancia
df_u = df_u.dropna(
    subset=["q0", "acx"]
).reset_index(drop=True)

df_u

,Time,t,dt,q0,q1,q2,q3,acx,acy,acz
0,1778882044327970304,0.000000,NaN,0.984476,-0.008141,-0.166982,0.053449,-0.328474,0.030808,-0.944354
1,1778882044337928704,0.009958,0.009958,0.984475,-0.008156,-0.166987,0.053454,-0.325551,0.033112,-0.940716
2,1778882044347867392,0.019897,0.009939,0.984475,-0.008152,-0.166986,0.053464,-0.325238,0.036366,-0.935931
3,1778882044357786368,0.029816,0.009919,0.984476,-0.008139,-0.166986,0.053453,-0.327034,0.033377,-0.937326
4,1778882044367685888,0.039716,0.009900,0.984477,-0.008143,-0.166981,0.053452,-0.327513,0.033002,-0.942016
...,...,...,...,...,...,...,...,...,...,...
944,1778882053767425536,9.439455,0.010173,0.984448,-0.008012,-0.167037,0.053824,-0.326244,0.033626,-0.939517
945,1778882053777578240,9.449608,0.010153,0.984448,-0.008013,-0.167036,0.053825,-0.326059,0.033698,-0.939841
946,1778882053787710720,9.459740,0.010132,0.984448,-0.008013,-0.167037,0.053824,-0.325983,0.033813,-0.939869
947,1778882053797822976,9.469853,0.010112,0.984447,-0.008016,-0.167038,0.053825,-0.326069,0.033642,-0.939342


In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation as R


# ============================================================
# 1. Leer CSV de SensorConnect desde DATA_START
# ============================================================

def read_quaternion_sensorconnect_csv(filepath):
    """
    Lee un CSV exportado por SensorConnect que contiene DATA_START.

    Formato esperado:
        DATA_START
        Time,
        inertial-xxxx:estOrientQuaternion[0-0],
        inertial-xxxx:estOrientQuaternion[0-1],
        inertial-xxxx:estOrientQuaternion[0-2],
        inertial-xxxx:estOrientQuaternion[0-3],
        inertial-xxxx:estOrientQuaternion:valid

    Retorna:
        DataFrame con columnas:
            Time, q0, q1, q2, q3, qValid, t
    """

    # Buscar fila donde aparece DATA_START
    with open(filepath, "r", encoding="utf-8") as f:
        lines = f.readlines()

    data_start_idx = None

    for i, line in enumerate(lines):
        if line.strip() == "DATA_START":
            data_start_idx = i
            break

    if data_start_idx is None:
        raise ValueError("No se encontró DATA_START en el archivo.")

    # La cabecera real está justo después de DATA_START
    header_idx = data_start_idx + 1

    df = pd.read_csv(filepath, skiprows=header_idx)

    # Limpiar nombres de columnas
    df.columns = df.columns.str.strip()

    # Detectar columnas de cuaternión
    q_cols_raw = [
        col for col in df.columns
        if "estOrientQuaternion[0-" in col
    ]

    q_cols_raw = sorted(q_cols_raw)

    if len(q_cols_raw) != 4:
        raise ValueError(
            f"No se encontraron 4 columnas de cuaternión. Encontradas: {q_cols_raw}"
        )

    # Detectar columna valid
    valid_cols = [
        col for col in df.columns
        if "estOrientQuaternion:valid" in col
    ]

    if len(valid_cols) == 0:
        raise ValueError("No se encontró columna estOrientQuaternion:valid.")

    valid_col = valid_cols[0]

    # Renombrar
    rename_map = {
        q_cols_raw[0]: "q0",
        q_cols_raw[1]: "q1",
        q_cols_raw[2]: "q2",
        q_cols_raw[3]: "q3",
        valid_col: "qValid",
    }

    df = df.rename(columns=rename_map)

    # Convertir a numérico
    cols_needed = ["Time", "q0", "q1", "q2", "q3", "qValid"]

    for col in cols_needed:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Eliminar filas inválidas o incompletas
    df = df.dropna(subset=cols_needed).copy()

    # Filtrar solo cuaterniones válidos
    df = df[df["qValid"] == 1].copy()

    # Eliminar cuaterniones cero
    q_norm = np.linalg.norm(df[["q0", "q1", "q2", "q3"]].to_numpy(), axis=1)
    df = df[q_norm > 1e-9].copy()

    # Normalizar cuaterniones
    q = df[["q0", "q1", "q2", "q3"]].to_numpy(dtype=float)
    q = q / np.linalg.norm(q, axis=1, keepdims=True)

    df["q0"] = q[:, 0]
    df["q1"] = q[:, 1]
    df["q2"] = q[:, 2]
    df["q3"] = q[:, 3]

    # Tiempo relativo en segundos
    df = df.sort_values("Time").reset_index(drop=True)
    df["t"] = (df["Time"] - df["Time"].iloc[0]) / 1e9

    return df


# ============================================================
# 2. Conversión de formato de cuaternión
# ============================================================

def wxyz_to_xyzw(q_wxyz):
    """
    Convierte cuaternión [w, x, y, z] a [x, y, z, w],
    que es el formato usado por scipy.spatial.transform.Rotation.
    """
    q_wxyz = np.asarray(q_wxyz, dtype=float)

    return np.array([
        q_wxyz[1],
        q_wxyz[2],
        q_wxyz[3],
        q_wxyz[0],
    ])


def xyzw_to_wxyz(q_xyzw):
    """
    Convierte cuaternión [x, y, z, w] a [w, x, y, z].
    """
    q_xyzw = np.asarray(q_xyzw, dtype=float)

    return np.array([
        q_xyzw[3],
        q_xyzw[0],
        q_xyzw[1],
        q_xyzw[2],
    ])


# ============================================================
# 3. Obtener cuaternión de referencia inicial
# ============================================================

def get_initial_reference_quaternion(df, method="first", initial_time_window=1.0):
    """
    Obtiene el cuaternión inicial de referencia.

    method:
        "first":
            usa el primer cuaternión válido.

        "mean":
            promedia los cuaterniones durante los primeros
            initial_time_window segundos.

    Retorna:
        scipy Rotation R_ref
    """

    if method == "first":
        q_ref_wxyz = df[["q0", "q1", "q2", "q3"]].iloc[0].to_numpy(dtype=float)
        q_ref_wxyz = q_ref_wxyz / np.linalg.norm(q_ref_wxyz)

        R_ref = R.from_quat(wxyz_to_xyzw(q_ref_wxyz))
        return R_ref

    elif method == "mean":
        df_init = df[df["t"] <= initial_time_window].copy()

        if len(df_init) == 0:
            raise ValueError("No hay muestras dentro de la ventana inicial.")

        q_init_wxyz = df_init[["q0", "q1", "q2", "q3"]].to_numpy(dtype=float)

        # Normalizar
        q_init_wxyz = q_init_wxyz / np.linalg.norm(q_init_wxyz, axis=1, keepdims=True)

        # Promedio simple con corrección de signo.
        # Los cuaterniones q y -q representan la misma orientación.
        # Se alinean todos al signo del primero.
        q0 = q_init_wxyz[0]

        for k in range(len(q_init_wxyz)):
            if np.dot(q0, q_init_wxyz[k]) < 0:
                q_init_wxyz[k] *= -1.0

        q_ref_wxyz = np.mean(q_init_wxyz, axis=0)
        q_ref_wxyz = q_ref_wxyz / np.linalg.norm(q_ref_wxyz)

        R_ref = R.from_quat(wxyz_to_xyzw(q_ref_wxyz))
        return R_ref

    else:
        raise ValueError("method debe ser 'first' o 'mean'.")


# ============================================================
# 4. Calibrar orientación absoluta a orientación relativa
# ============================================================

def calibrate_quaternion_to_relative(
    df,
    reference_method="first",
    initial_time_window=2.0,
    # euler_sequence="xyz"
):
    """
    Calcula cuaterniones relativos:

        q_rel(t) = q_ref^{-1} ⊗ q(t)

    Hace que la orientación inicial sea aproximadamente:

        [1, 0, 0, 0]

    Entrada:
        df con columnas q0, q1, q2, q3 en formato [w,x,y,z].

    Salida:
        df con columnas:
            q0_rel, q1_rel, q2_rel, q3_rel
            roll_rel_deg, pitch_rel_deg, yaw_rel_deg
    """

    df = df.copy()

    R_ref = get_initial_reference_quaternion(
        df,
        method=reference_method,
        initial_time_window=initial_time_window
    )
    
    q_rel_list = []
    euler_rel_list = []

    for _, row in df.iterrows():
        q_wxyz = row[["q0", "q1", "q2", "q3"]].to_numpy(dtype=float)
        q_wxyz = q_wxyz / np.linalg.norm(q_wxyz)

        R_current = R.from_quat(wxyz_to_xyzw(q_wxyz))

        # Calibración relativa
        R_rel = R_ref.inv() * R_current

        q_rel_xyzw = R_rel.as_quat()
        q_rel_wxyz = xyzw_to_wxyz(q_rel_xyzw)

        q_rel_list.append(q_rel_wxyz)

        # euler_rel_deg = R_rel.as_euler(euler_sequence, degrees=True)
        # euler_rel_list.append(euler_rel_deg)

    q_rel = np.array(q_rel_list)
    # euler_rel = np.array(euler_rel_list)

    df["q0_rel"] = q_rel[:, 0]
    df["q1_rel"] = q_rel[:, 1]
    df["q2_rel"] = q_rel[:, 2]
    df["q3_rel"] = q_rel[:, 3]

    # df["roll_rel_deg"] = euler_rel[:, 0]
    # df["pitch_rel_deg"] = euler_rel[:, 1]
    # df["yaw_rel_deg"] = euler_rel[:, 2]

    return df, R_ref


# ============================================================
# 7. Main
# ============================================================

if __name__ == "__main__":

    # filepath = r"calibracion_orientacion\cuat_3.csv"

    # df = read_quaternion_sensorconnect_csv(filepath)

    df = df_u

    print("\nCuaternión inicial original [w,x,y,z]:")
    print(df[["q0", "q1", "q2", "q3"]].iloc[0].to_numpy())

    # Calibrar orientación inicial a identidad
    df_rel, R_ref  = calibrate_quaternion_to_relative(
        df,
        reference_method="mean",       # "first" o "mean"
        initial_time_window=2.0,
        # euler_sequence="xyz"
    )

    print("\nPrimer cuaternión relativo [w,x,y,z]:")
    print(df_rel[["q0_rel", "q1_rel", "q2_rel", "q3_rel"]].iloc[0].to_numpy())

    # Guardar resultado
    output_file = "quaternions_relative_calibrated.csv"
    df_rel.to_csv(output_file, index=False)

    print(f"\nArchivo guardado: {output_file}")

    print(R_ref)


Cuaternión inicial original [w,x,y,z]:
[ 0.984476 -0.008141 -0.166982  0.053449]

Primer cuaternión relativo [w,x,y,z]:
[ 9.99999998e-01 -5.72593596e-05  2.21191068e-05 -3.23050515e-05]

Archivo guardado: quaternions_relative_calibrated.csv
Rotation.from_matrix(array([[ 0.93849916, -0.1026182 , -0.32967988],
                            [ 0.10802157,  0.99414666, -0.00193943],
                            [ 0.32794918, -0.03379238,  0.94409079]]))


In [7]:
import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R


def normalize_vector(v, eps=1e-12):
    """
    Normaliza un vector 3D.
    """
    n = np.linalg.norm(v)
    if n < eps:
        raise ValueError("Vector con norma casi cero, no se puede normalizar.")
    return v / n


def rotation_from_two_vectors(a, b):
    """
    Retorna una matriz R tal que aproximadamente:

        R @ a = b

    donde a y b son vectores 3D unitarios.

    Usa la fórmula de Rodrigues.
    """
    a = normalize_vector(np.asarray(a, dtype=float))
    b = normalize_vector(np.asarray(b, dtype=float))

    v = np.cross(a, b)
    c = np.dot(a, b)
    s = np.linalg.norm(v)

    # Caso: vectores casi iguales
    if s < 1e-12 and c > 0:
        return np.eye(3)

    # Caso: vectores opuestos
    if s < 1e-12 and c < 0:
        # Elegir un eje perpendicular arbitrario
        axis = np.array([1.0, 0.0, 0.0])
        if abs(a[0]) > 0.9:
            axis = np.array([0.0, 1.0, 0.0])

        axis = normalize_vector(np.cross(a, axis))
        return R.from_rotvec(np.pi * axis).as_matrix()

    vx = np.array([
        [0,      -v[2],   v[1]],
        [v[2],    0,    -v[0]],
        [-v[1],  v[0],   0]
    ])

    R_mat = np.eye(3) + vx + vx @ vx * ((1 - c) / (s ** 2))

    return R_mat


def calibrate_static_roll_pitch(
    df,
    acc_cols=("acx", "acy", "acz"),
    gyro_cols=None,
    time_col="t",
    t_start=None,
    t_end=None,
    zupt_flag_col=None,
    expected_specific_force_vehicle=np.array([0.0, 0.0, -1.0]),
    gyro_threshold=None,
    verbose=True
):
    """
    Calibración estática de roll/pitch IMU -> vehículo usando fuerza específica.

    Parámetros
    ----------
    df:
        DataFrame con aceleraciones del IMU.
    
    acc_cols:
        Columnas de acelerómetro que contienen fuerza específica.
        Ejemplo:
            ("scaledAccelX", "scaledAccelY", "scaledAccelZ")
        o:
            ("ax", "ay", "az")

    gyro_cols:
        Opcional. Columnas de giroscopio para validar reposo.
        Ejemplo:
            ("gx", "gy", "gz")

    time_col:
        Columna de tiempo en segundos.

    t_start, t_end:
        Ventana estática manual.

    zupt_flag_col:
        Opcional. Si existe, usa filas donde zupt_flag == 1.

    expected_specific_force_vehicle:
        Vector esperado de fuerza específica en marco vehículo.
        Para vehículo con Z positivo hacia abajo:
            [0, 0, -1]
        Para vehículo con Z positivo hacia arriba:
            [0, 0, +1]

    gyro_threshold:
        Opcional. Umbral de norma giroscópica para filtrar muestras quietas.

    Retorna
    -------
    result:
        Diccionario con:
        - R_vb_rp: matriz de rotación IMU -> vehículo, solo roll/pitch
        - q_vb_rp_xyzw: cuaternión scipy [x, y, z, w]
        - f_b_mean: promedio medido en IMU
        - f_b_unit: dirección medida normalizada
        - f_v_expected_unit: dirección esperada normalizada
        - f_v_after: fuerza específica promedio rotada al vehículo
        - roll_pitch_calib_euler_deg: euler aproximado de la corrección
    """

    work = df.copy()

    # 1. Filtrar por ventana de tiempo
    mask = np.ones(len(work), dtype=bool)

    if t_start is not None:
        mask &= work[time_col].to_numpy() >= t_start

    if t_end is not None:
        mask &= work[time_col].to_numpy() <= t_end

    # 2. Filtrar por ZUPT flag
    if zupt_flag_col is not None and zupt_flag_col in work.columns:
        mask &= work[zupt_flag_col].to_numpy().astype(bool)

    # 3. Filtrar por giroscopio bajo
    if gyro_cols is not None and gyro_threshold is not None:
        gyro = work[list(gyro_cols)].to_numpy(dtype=float)
        gyro_norm = np.linalg.norm(gyro, axis=1)
        mask &= gyro_norm < gyro_threshold

    static_df = work.loc[mask].copy()

    if len(static_df) < 10:
        raise ValueError("Muy pocas muestras estáticas para calibrar.")

    # 4. Promedio de fuerza específica en IMU
    f_b_samples = static_df[list(acc_cols)].to_numpy(dtype=float)
    f_b_mean = np.mean(f_b_samples, axis=0)
    f_b_unit = normalize_vector(f_b_mean)

    # 5. Vector esperado en vehículo
    f_v_expected_unit = normalize_vector(expected_specific_force_vehicle)

    # 6. Rotación IMU -> vehículo que alinea f_b con f_v_expected
    R_vb_rp = rotation_from_two_vectors(f_b_unit, f_v_expected_unit)

    # 7. Validación: cómo queda el promedio después de rotar
    f_v_after = R_vb_rp @ f_b_mean

    # 8. Cuaternión equivalente
    q_vb_rp_xyzw = R.from_matrix(R_vb_rp).as_quat()

    # 9. Euler aproximado de la corrección
    euler_deg = R.from_matrix(R_vb_rp).as_euler("xyz", degrees=True)

    result = {
        "R_vb_rp": R_vb_rp,
        "q_vb_rp_xyzw": q_vb_rp_xyzw,
        "f_b_mean": f_b_mean,
        "f_b_unit": f_b_unit,
        "f_v_expected_unit": f_v_expected_unit,
        "f_v_after": f_v_after,
        "roll_pitch_calib_euler_deg_xyz": euler_deg,
        "num_static_samples": len(static_df),
        "static_df": static_df,
    }

    if verbose:
        print("R_vb_rp: ", R_vb_rp)
        print("Muestras usadas:", len(static_df))
        print("f_b_mean IMU:", f_b_mean)
        print("||f_b_mean||:", np.linalg.norm(f_b_mean))
        print("f_b_unit IMU:", f_b_unit)
        print("f_v_expected_unit:", f_v_expected_unit)
        print("f_v_after:", f_v_after)
        print("Euler corrección xyz [deg]:", euler_deg)
        print("q_vb_rp scipy [x, y, z, w]:", q_vb_rp_xyzw)

    return result

In [8]:
calib = calibrate_static_roll_pitch(
    df,
    acc_cols=("acx", "acy", "acz"),
    time_col="t",
    t_start=None,
    t_end=5,
    expected_specific_force_vehicle=np.array([0.0, 0.0, -1.0]),
    verbose=True
)

R_vb_rp = calib["R_vb_rp"]
q_vb_rp = calib["q_vb_rp_xyzw"]

R_vb_rp:  [[ 0.94470223  0.00570843 -0.32787971]
 [ 0.00570843  0.99941071  0.03384728]
 [ 0.32787971 -0.03384728  0.94411295]]
Muestras usadas: 501
f_b_mean IMU: [-0.32622662  0.03367663 -0.93935296]
||f_b_mean||: 0.9949582420565747
f_b_unit IMU: [-0.32787971  0.03384728 -0.94411295]
f_v_expected_unit: [ 0.  0. -1.]
f_v_after: [-3.11299061e-17  4.14051935e-18 -9.94958242e-01]
Euler corrección xyz [deg]: [ -2.05322502 -19.1401332    0.34620975]
q_vb_rp scipy [x, y, z, w]: [-0.01716517 -0.16627954  0.          0.98592924]


In [9]:
print(R_vb_rp)
print(q_vb_rp)

[[ 0.94470223  0.00570843 -0.32787971]
 [ 0.00570843  0.99941071  0.03384728]
 [ 0.32787971 -0.03384728  0.94411295]]
[-0.01716517 -0.16627954  0.          0.98592924]


In [10]:
df

,Time,t,dt,q0,q1,q2,q3,acx,acy,acz
0,1778882044327970304,0.000000,NaN,0.984476,-0.008141,-0.166982,0.053449,-0.328474,0.030808,-0.944354
1,1778882044337928704,0.009958,0.009958,0.984475,-0.008156,-0.166987,0.053454,-0.325551,0.033112,-0.940716
2,1778882044347867392,0.019897,0.009939,0.984475,-0.008152,-0.166986,0.053464,-0.325238,0.036366,-0.935931
3,1778882044357786368,0.029816,0.009919,0.984476,-0.008139,-0.166986,0.053453,-0.327034,0.033377,-0.937326
4,1778882044367685888,0.039716,0.009900,0.984477,-0.008143,-0.166981,0.053452,-0.327513,0.033002,-0.942016
...,...,...,...,...,...,...,...,...,...,...
944,1778882053767425536,9.439455,0.010173,0.984448,-0.008012,-0.167037,0.053824,-0.326244,0.033626,-0.939517
945,1778882053777578240,9.449608,0.010153,0.984448,-0.008013,-0.167036,0.053825,-0.326059,0.033698,-0.939841
946,1778882053787710720,9.459740,0.010132,0.984448,-0.008013,-0.167037,0.053824,-0.325983,0.033813,-0.939869
947,1778882053797822976,9.469853,0.010112,0.984447,-0.008016,-0.167038,0.053825,-0.326069,0.033642,-0.939342


In [11]:
df = df.copy()
df

,Time,t,dt,q0,q1,q2,q3,acx,acy,acz
0,1778882044327970304,0.000000,NaN,0.984476,-0.008141,-0.166982,0.053449,-0.328474,0.030808,-0.944354
1,1778882044337928704,0.009958,0.009958,0.984475,-0.008156,-0.166987,0.053454,-0.325551,0.033112,-0.940716
2,1778882044347867392,0.019897,0.009939,0.984475,-0.008152,-0.166986,0.053464,-0.325238,0.036366,-0.935931
3,1778882044357786368,0.029816,0.009919,0.984476,-0.008139,-0.166986,0.053453,-0.327034,0.033377,-0.937326
4,1778882044367685888,0.039716,0.009900,0.984477,-0.008143,-0.166981,0.053452,-0.327513,0.033002,-0.942016
...,...,...,...,...,...,...,...,...,...,...
944,1778882053767425536,9.439455,0.010173,0.984448,-0.008012,-0.167037,0.053824,-0.326244,0.033626,-0.939517
945,1778882053777578240,9.449608,0.010153,0.984448,-0.008013,-0.167036,0.053825,-0.326059,0.033698,-0.939841
946,1778882053787710720,9.459740,0.010132,0.984448,-0.008013,-0.167037,0.053824,-0.325983,0.033813,-0.939869
947,1778882053797822976,9.469853,0.010112,0.984447,-0.008016,-0.167038,0.053825,-0.326069,0.033642,-0.939342


In [12]:
import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R

# ============================================================
# 1. Matriz de rotación fija
# ============================================================

# R_fix_matrix = np.array([
#     [ 9.40965430e-01,  9.28686179e-03,  3.38375257e-01],
#     [-4.50886022e-04,  9.99657086e-01, -2.61822011e-02],
#     [-3.38502374e-01,  2.44839775e-02,  9.40646946e-01]
# ])

# R_fix = R.from_matrix(R_fix_matrix)

# print(R_fix)

# ============================================================
# 2. Función para aplicar la rotación a cada cuaternión
# ============================================================

def apply_fixed_rotation_to_quaternions(
    df,
    quat_cols=("q0", "q1", "q2", "q3"),
    output_cols=("q0_rot", "q1_rot", "q2_rot", "q3_rot"),
    # order="left"
):
    """
    Aplica una rotación fija R_fix a los cuaterniones del DataFrame.

    Entrada:
        q0,q1,q2,q3 en formato [w, x, y, z]

    SciPy usa:
        [x, y, z, w]

    order:
        "left"  -> q_new = R_fix * q_original
        "right" -> q_new = q_original * R_fix
    """

    q0, q1, q2, q3 = quat_cols

    # Pasar de [w,x,y,z] a [x,y,z,w]
    q_scipy = df[[q1, q2, q3, q0]].to_numpy()

    # Normalizar por seguridad
    q_scipy = q_scipy / np.linalg.norm(q_scipy, axis=1, keepdims=True)

    R_original = R.from_quat(q_scipy)

    # if order == "left":
    #     R_new = R_fix * R_original
    # elif order == "right":
    #     R_new = R_original * R_fix
    # else:
    #     raise ValueError("order debe ser 'left' o 'right'")

    # Calibración relativa
    R_rel = R_ref.inv() * R_original
 
    # Volver de SciPy [x,y,z,w] a [w,x,y,z]
    q_new_scipy = R_rel.as_quat()
    q_new_wxyz = np.column_stack([
        q_new_scipy[:, 3],
        q_new_scipy[:, 0],
        q_new_scipy[:, 1],
        q_new_scipy[:, 2],
    ])

    R_rel

    df_out = df.copy()
    df_out[list(output_cols)] = q_new_wxyz

    return df_out

In [13]:
df_rot = apply_fixed_rotation_to_quaternions(
    df,
    quat_cols=("q0", "q1", "q2", "q3"),
    output_cols=("q0_cal", "q1_cal", "q2_cal", "q3_cal"),
    # order="left"
)

df_rot.tail()

,Time,t,dt,q0,q1,q2,q3,acx,acy,acz,q0_cal,q1_cal,q2_cal,q3_cal
944,1778882053767425536,9.439455,0.010173,0.984448,-0.008012,-0.167037,0.053824,-0.326244,0.033626,-0.939517,1.0,0.000129,-0.000047,0.000316
945,1778882053777578240,9.449608,0.010153,0.984448,-0.008013,-0.167036,0.053825,-0.326059,0.033698,-0.939841,1.0,0.000128,-0.000046,0.000318
946,1778882053787710720,9.459740,0.010132,0.984448,-0.008013,-0.167037,0.053824,-0.325983,0.033813,-0.939869,1.0,0.000128,-0.000047,0.000317
947,1778882053797822976,9.469853,0.010112,0.984447,-0.008016,-0.167038,0.053825,-0.326069,0.033642,-0.939342,1.0,0.000125,-0.000048,0.000318
948,1778882053807915520,9.479945,0.010093,0.984448,-0.008017,-0.167037,0.053826,-0.326305,0.033512,-0.939374,1.0,0.000125,-0.000046,0.000319


In [14]:
import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R


# ============================================================
# Rotación fija IMU/body -> vehículo
# ============================================================

# R_vb = np.array([
#     [ 0.94089068,  0.00425876,  0.33868363],
#     [ 0.00425876,  0.99969316, -0.02440178],
#     [-0.33868363,  0.02440178,  0.94058384]
# ])

# r_vb = R.from_matrix(R_vb)

r_vb = R.from_matrix(R_vb_rp)

# Inversa: vehículo -> IMU/body
r_bv = r_vb.inv()


# ============================================================
# Función para aplicar rotación de instalación
# ============================================================

def apply_vehicle_mount_rotation(
    df,
    quat_cols=("q0_cal", "q1_cal", "q2_cal", "q3_cal"),
    acc_cols=("acx", "acy", "acz"),
    prefix_out="veh"
):
    df = df.copy()

    q0, q1, q2, q3 = quat_cols

    # --------------------------------------------------------
    # 1. Leer cuaterniones calibrados del IMU
    #    Entrada: [w, x, y, z]
    #    SciPy:   [x, y, z, w]
    # --------------------------------------------------------
    q_wxyz = df[[q0, q1, q2, q3]].to_numpy(dtype=float)

    # Normalizar por seguridad
    q_norm = np.linalg.norm(q_wxyz, axis=1, keepdims=True)
    q_wxyz = q_wxyz / q_norm

    q_xyzw = np.column_stack([
        q_wxyz[:, 1],
        q_wxyz[:, 2],
        q_wxyz[:, 3],
        q_wxyz[:, 0],
    ])

    r_nb = R.from_quat(q_xyzw)

    # --------------------------------------------------------
    # 2. Convertir orientación IMU -> orientación vehículo
    #
    # Si q_cal representa R_nb:
    #
    #   R_nv = R_nb @ R_bv
    #
    # En SciPy:
    #
    #   r_nv = r_nb * r_bv
    # --------------------------------------------------------
    r_nv = r_nb * r_bv

    q_nv_xyzw = r_nv.as_quat()

    # Regresar a formato [w, x, y, z]
    df[f"q0_{prefix_out}"] = q_nv_xyzw[:, 3]
    df[f"q1_{prefix_out}"] = q_nv_xyzw[:, 0]
    df[f"q2_{prefix_out}"] = q_nv_xyzw[:, 1]
    df[f"q3_{prefix_out}"] = q_nv_xyzw[:, 2]

    # --------------------------------------------------------
    # 3. Rotar aceleración del IMU al vehículo
    #
    #   a_v = R_vb @ a_b
    # --------------------------------------------------------
    ax, ay, az = acc_cols
    a_b = df[[ax, ay, az]].to_numpy(dtype=float)

    # a_v = (R_vb @ a_b.T).T

    a_v = (R_vb_rp @ a_b.T).T

    df[f"a{prefix_out}_x"] = a_v[:, 0]
    df[f"a{prefix_out}_y"] = a_v[:, 1]
    df[f"a{prefix_out}_z"] = a_v[:, 2]

    return df

In [15]:
df2 = apply_vehicle_mount_rotation(
    df_rot,
    quat_cols=("q0_cal", "q1_cal", "q2_cal", "q3_cal"),
    acc_cols=("acx", "acy", "acz"),
    prefix_out="veh"
)

df2

,Time,t,dt,q0,q1,q2,q3,acx,acy,acz,...,q1_cal,q2_cal,q3_cal,q0_veh,q1_veh,q2_veh,q3_veh,aveh_x,aveh_y,aveh_z
0,1778882044327970304,0.000000,NaN,0.984476,-0.008141,-0.166982,0.053449,-0.328474,0.030808,-0.944354,...,-0.000057,0.000022,-0.000032,0.985927,0.017114,0.166301,-0.000042,-0.000500,-0.003049,-1.000320
1,1778882044337928704,0.009958,0.009958,0.984475,-0.008156,-0.166987,0.053454,-0.325551,0.033112,-0.940716,...,-0.000071,0.000018,-0.000025,0.985928,0.017099,0.166297,-0.000037,0.001082,-0.000607,-0.996004
2,1778882044347867392,0.019897,0.009939,0.984475,-0.008152,-0.166986,0.053464,-0.325238,0.036366,-0.935931,...,-0.000066,0.000018,-0.000016,0.985927,0.017103,0.166297,-0.000027,-0.000173,0.002809,-0.991494
3,1778882044357786368,0.029816,0.009919,0.984476,-0.008139,-0.166986,0.053453,-0.327034,0.033377,-0.937326,...,-0.000055,0.000018,-0.000029,0.985927,0.017116,0.166297,-0.000038,-0.001429,-0.000235,-0.993299
4,1778882044367685888,0.039716,0.009900,0.984477,-0.008143,-0.166981,0.053452,-0.327513,0.033002,-0.942016,...,-0.000059,0.000023,-0.000029,0.985926,0.017112,0.166302,-0.000039,-0.000346,-0.000772,-0.997872
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
944,1778882053767425536,9.439455,0.010173,0.984448,-0.008012,-0.167037,0.053824,-0.326244,0.033626,-0.939517,...,0.000129,-0.000047,0.000316,0.985935,0.017240,0.166239,0.000334,0.000037,-0.000057,-0.995117
945,1778882053777578240,9.449608,0.010153,0.984448,-0.008013,-0.167036,0.053825,-0.326059,0.033698,-0.939841,...,0.000128,-0.000046,0.000318,0.985935,0.017239,0.166240,0.000335,0.000319,0.000005,-0.995365
946,1778882053787710720,9.459740,0.010132,0.984448,-0.008013,-0.167037,0.053824,-0.325983,0.033813,-0.939869,...,0.000128,-0.000047,0.000317,0.985935,0.017239,0.166239,0.000334,0.000400,0.000120,-0.995370
947,1778882053797822976,9.469853,0.010112,0.984447,-0.008016,-0.167038,0.053825,-0.326069,0.033642,-0.939342,...,0.000125,-0.000048,0.000318,0.985935,0.017236,0.166238,0.000335,0.000145,-0.000034,-0.994895
